In [11]:
from snack_stack_graph import build_graph
import uuid
from tools.config import get_llm
from tools.menu import MENU_COLLECTION_NAME, load_menu_documents
from tools.orders import load_orders_documents
from tools.vector_store import VectorStore

# Initialize the Graph

In [2]:
MENU_COLLECTION_NAME = "menu_collection"
orders = load_orders_documents("../../data/orders.json")
persist_directory = "../../data/chroma_store_3"
documents = load_menu_documents("../../data/menu.json")
vector_store = VectorStore(persist_directory=persist_directory, collection_name=MENU_COLLECTION_NAME)
menu_store = vector_store.get_create_collection(documents)

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 314/314 [00:00<00:00, 5526.12it/s]


In [3]:
llm = get_llm()

In [4]:
graph, context = build_graph(orders=orders, menu_collection=menu_store, llm=llm)

In [12]:
MENU_AGENT_NODE = "menu_agent_node"
MENU_AGENT_TOOL_NODE = "menu_agent_tool_node"
ORDER_AGENT_NODE = "order_agent_node"
ORDER_AGENT_TOOL_NODE = "order_agent_tool_node"
SYNTHESIZER_AGENT_NODE = "synthesizer_agent_node"
ORCHESTRATOR_AGENT_NODE = "orchestrator_agent"

def print_node_output_field(field_name:str, node_output:dict):
    if field_name in node_output:
        print(f"{field_name} {node_output[field_name]}\n")

def chat_with_user(prompt:str, user:str):
    input = {
        "user_input": prompt,
        "messages": [],
        "output": "",
        "route": ""
    }
    config={"configurable": {"thread_id": str(uuid.uuid4())}}
    
    for step in graph.stream(input, config, context=context):
        for node_name, node_output in step.items():
             print(f"\n--- Using: {node_name} ---")
             print(node_output)
        #     if node_name == ORCHESTRATOR_AGENT_NODE:
        #         for task in node_output['tasks']:
        #             print(f"\n\ndispatching to agent {task.agent} with description: {task.description}")
        #         print_node_output_field('messages', node_output)
        #         print_node_output_field('requires_synthesis', node_output)
        #         print("\n----------------------------")
        #     elif node_name == MENU_AGENT_NODE:
        #         print_node_output_field('messages', node_output)
        #         print_node_output_field('menu_agent_output', node_output)
        #         print("\n----------------------------")
        #     elif node_name == MENU_AGENT_TOOL_NODE:
        #         print_node_output_field('messages', node_output)
        #         print("\n----------------------------")

                    
  #       stream_and_save_response(f"Route: {node_output["route"]}")
  #     elif node_name in [BLOG_WRITER_TOOL_NODE, SOCIAL_MEDIA_WRITER_NODE]:
  #       for msg in node_output.get("messages", []):
  #         stream_and_save_response(f"Tool Result: {str(msg.content)[:200000]}...")  
  #     else:
  #       stream_and_save_response(f"{node_output["output"]}")

In [13]:
user = "user_123"
chat_with_user("Can you recommend some italian food",user)


###orchestrator_node###

messages: []

user_input: Can you recommend some italian food

menu_agent_messages: []

order_agent_messages: []

###dispatch_to_agents###

messages: []

user_input: Can you recommend some italian food

tasks: [AgentTask(agent='menu_agent_node', description='User is asking for recommendations on Italian food. Please provide Italian food recommendations from the menu.')]

requires_synthesis: False

menu_agent_messages: []

order_agent_messages: []

--- Using: orchestrator_agent ---
{'tasks': [AgentTask(agent='menu_agent_node', description='User is asking for recommendations on Italian food. Please provide Italian food recommendations from the menu.')], 'requires_synthesis': False, 'user_input': 'Can you recommend some italian food'}

###menu_agent_node###

messages: []

user_input: Can you recommend some italian food

tasks: [AgentTask(agent='menu_agent_node', description='User is asking for recommendations on Italian food. Please provide Italian food recommend

/Users/elsadunsmoor/projects/snack-stack-ai/.venv/lib/python3.14/site-packages/pydantic/functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ContextSchema(orders=<mul...input at 0x13cc700f0>)]), input_type=ContextSchema])
  function=lambda v, h: h(v), schema=original_schema
/Users/elsadunsmoor/projects/snack-stack-ai/.venv/lib/python3.14/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ContextSchema(orders=<mul...input at 0x13cc700f0>)]), input_type=ContextSchema])
  return self.__pydantic_serializer__.to_python(



--- Using: menu_agent_tool_node ---
{'menu_agent_messages': [ToolMessage(content='Top 3 matches for Italian food:\n\nDish: Aglio e Olio\n        Cuisine: Italian\n        Price: 279\n        Rating: 4.5\n        Dietary: Vegan\n        Description: Spaghetti with garlic, chilli, olive oil, parsley\n---\nDish: Vegan Pasta Primavera\n        Cuisine: Italian\n        Price: 349\n        Rating: 4.5\n        Dietary: Vegan\n        Description: Penne with seasonal vegetables, olive oil, garlic\n---\nDish: Margherita Pizza\n        Cuisine: Italian\n        Price: 299\n        Rating: 4.7\n        Dietary: Veg\n        Description: Classic thin crust with tomato, mozzarella, basil\n---\n', name='search_menu_catalog', tool_call_id='toolu_01Q734EQg1sVUPe53Y98qpoF')]}

###menu_agent_node###

messages: []

user_input: Can you recommend some italian food

tasks: [AgentTask(agent='menu_agent_node', description='User is asking for recommendations on Italian food. Please provide Italian food reco

In [7]:
menu_store.similarity_search("Margherita Pizza")

[Document(id='d973bcfd-f1ad-484f-91bf-8b42f5cfc1de', metadata={'Cuisine': 'Italian', 'Dietary': 'Veg', 'Dish': 'Margherita Pizza'}, page_content='Dish: Margherita Pizza\n        Cuisine: Italian\n        Price: 299\n        Rating: 4.7\n        Dietary: Veg\n        Description: Classic thin crust with tomato, mozzarella, basil'),
 Document(id='6e633c3a-bd82-415a-a882-5b4bda431d11', metadata={'Dietary': 'Vegan', 'Dish': 'Aglio e Olio', 'Cuisine': 'Italian'}, page_content='Dish: Aglio e Olio\n        Cuisine: Italian\n        Price: 279\n        Rating: 4.5\n        Dietary: Vegan\n        Description: Spaghetti with garlic, chilli, olive oil, parsley'),
 Document(id='0fc8e3cd-c03b-4831-ad02-83908394c64e', metadata={'Dietary': 'Vegan', 'Cuisine': 'Italian', 'Dish': 'Vegan Pasta Primavera'}, page_content='Dish: Vegan Pasta Primavera\n        Cuisine: Italian\n        Price: 349\n        Rating: 4.5\n        Dietary: Vegan\n        Description: Penne with seasonal vegetables, olive oil, g

In [8]:
menu_store.get()

{'ids': ['d973bcfd-f1ad-484f-91bf-8b42f5cfc1de',
  '0fc8e3cd-c03b-4831-ad02-83908394c64e',
  '2eb02f1e-44fc-4bf7-bebe-06eeacb26174',
  'd68c4fec-2396-4741-bd96-70722de36046',
  '45e506ff-f289-4681-ad1f-d736e1376708',
  'cafeb96f-2cea-43c9-95f7-d6fc2fff9b63',
  '6e633c3a-bd82-415a-a882-5b4bda431d11',
  '0980c471-5663-47ec-9ae2-40dfb888e8d6'],
 'embeddings': None,
 'documents': ['Dish: Margherita Pizza\n        Cuisine: Italian\n        Price: 299\n        Rating: 4.7\n        Dietary: Veg\n        Description: Classic thin crust with tomato, mozzarella, basil',
  'Dish: Vegan Pasta Primavera\n        Cuisine: Italian\n        Price: 349\n        Rating: 4.5\n        Dietary: Vegan\n        Description: Penne with seasonal vegetables, olive oil, garlic',
  'Dish: Butter Chicken\n        Cuisine: Indian\n        Price: 379\n        Rating: 4.9\n        Dietary: GF\n        Description: Creamy tomato curry with tender chicken and naan',
  'Dish: Vegan Buddha Bowl\n        Cuisine: Fusion\n

In [9]:
user = "user_123"
chat_with_user("what's the order status for order 124",user)

Deserializing unregistered type agents.state.AgentTask from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('agents.state', 'AgentTask')]



###orchestrator_node###

messages: [AIMessage(content='Great! Here are my top 3 Italian food recommendations from our menu:\n\n1. **Margherita Pizza** - $299 (Rating: 4.7/5) ⭐\n   - Classic thin crust with tomato, mozzarella, and basil\n   - Vegetarian option\n\n2. **Aglio e Olio** - $279 (Rating: 4.5/5)\n   - Spaghetti with garlic, chilli, olive oil, and parsley\n   - Vegan option\n\n3. **Vegan Pasta Primavera** - $349 (Rating: 4.5/5)\n   - Penne with seasonal vegetables, olive oil, and garlic\n   - Vegan option\n\nAll three dishes are highly rated! The Margherita Pizza has the highest rating and is a classic Italian favorite. Would you like more details about any of these dishes, or would you like me to search for other Italian options?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

user_input: what's the order status for order 124

tasks: [AgentTask(agent='menu_agent_node', description='User is asking for recommendations on Italian food. Pleas

/Users/elsadunsmoor/projects/snack-stack-ai/.venv/lib/python3.14/site-packages/pydantic/functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ContextSchema(orders=<mul...input at 0x13cc700f0>)]), input_type=ContextSchema])
  function=lambda v, h: h(v), schema=original_schema
/Users/elsadunsmoor/projects/snack-stack-ai/.venv/lib/python3.14/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ContextSchema(orders=<mul...input at 0x13cc700f0>)]), input_type=ContextSchema])
  return self.__pydantic_serializer__.to_python(



### result content="I was unable to find any order associated with order ID 124. This order number has no associated orders in our system. \n\nCould you please verify the order ID? If you have a different order ID, tracking number, or the email address associated with the order, I'd be happy to help you look it up." additional_kwargs={} response_metadata={'id': 'msg_011Ce8n2fWCySu8JMvBnXw59', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 920, 'output_tokens': 69, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'} id='lc_run--01a010ef-a7be-70d3-bcc4-d08320737319-0' tool_calls=[] invalid_tool_ca

In [10]:
user="user_123"
chat_with_user("the email for the order is arjun@example.com", user)


###orchestrator_node###

messages: [AIMessage(content='Great! Here are my top 3 Italian food recommendations from our menu:\n\n1. **Margherita Pizza** - $299 (Rating: 4.7/5) ⭐\n   - Classic thin crust with tomato, mozzarella, and basil\n   - Vegetarian option\n\n2. **Aglio e Olio** - $279 (Rating: 4.5/5)\n   - Spaghetti with garlic, chilli, olive oil, and parsley\n   - Vegan option\n\n3. **Vegan Pasta Primavera** - $349 (Rating: 4.5/5)\n   - Penne with seasonal vegetables, olive oil, and garlic\n   - Vegan option\n\nAll three dishes are highly rated! The Margherita Pizza has the highest rating and is a classic Italian favorite. Would you like more details about any of these dishes, or would you like me to search for other Italian options?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), AIMessage(content="# Order Status Inquiry - Order 124\n\n## Summary\n\nI was unable to locate order 124 in our system. The Orders Agent found **no associated orders*

/Users/elsadunsmoor/projects/snack-stack-ai/.venv/lib/python3.14/site-packages/pydantic/functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ContextSchema(orders=<mul...input at 0x13cc700f0>)]), input_type=ContextSchema])
  function=lambda v, h: h(v), schema=original_schema
/Users/elsadunsmoor/projects/snack-stack-ai/.venv/lib/python3.14/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ContextSchema(orders=<mul...input at 0x13cc700f0>)]), input_type=ContextSchema])
  return self.__pydantic_serializer__.to_python(


ValueError: Received multiple non-consecutive system messages.

In [17]:
#cleanup
import shutil
vector_store.delete_collection()
vector_store.close()
shutil.rmtree(persist_directory)